In [ ]:
using Pkg
Pkg.activate(".")

In [ ]:
using Revise
using RVSDDP
using Random
using Plots
using Gurobi
using CSV
using DataFrames
using Statistics
using Distributions
using LaTeXStrings
# using HiGHS
# optimizer = () -> HiGHS.Optimizer()
using Gurobi
const GRB_ENV = Gurobi.Env()
optimizer=() -> Gurobi.Optimizer(GRB_ENV)

In [ ]:
folder_list = Dict("RV-SDDP" => "shift_update_random_forward_0_parallel_1", "Cyclic SDDP" => "no_shift_0_parallel_1")

In [ ]:
results = Dict()
seed_list = 1:10
time_max = 36000
parallel = 1
time_list_oos = [300, 600,900,1200,1800,3600,7200,10800,18000, 36000]
discount_factor_list=[0.8, 0.99, 0.995]
period = 12
N_list = Dict(0.8 => 50000, 0.99 => 10000, 0.995 => 5000)
time_list_value = 300:300:36000
for discount_factor in discount_factor_list
      results[discount_factor] = Dict()
      N = N_list[discount_factor]
      for seed in seed_list
            for (label,folder) in folder_list
                  TimeHorizon = 12*Int(ceil(log(0.001)/(12*log(discount_factor))))
                  folder = "results_msppy/$(folder)/$(discount_factor)/seed_$(seed)_time_$(time_max)"
                  if isdir("$(folder)/oos")
                        oos_df = [CSV.read(joinpath("$(folder)/oos", "oos_horizon_$(time)_$(TimeHorizon)_$N.csv"), DataFrame) for time in time_list_oos]
                        oos_mean = [mean(oos_df[i][1:N, :oos_horizon])/1000 for i in 1:length(oos_df)]
                        oos_std = [std(oos_df[i][1:N, :oos_horizon])/1000 for i in 1:length(oos_df)]
                  else
                        oos_mean = [0.0 for _ in time_list_oos]
                        oos_std = [0.0 for _ in time_list_oos]
                  end
                  if isdir("$(folder)/active_cuts")
                        active_cuts_df = [CSV.read(joinpath("$(folder)/active_cuts", "time_$(time).csv"), DataFrame) for time in time_list_oos]
                        active_cuts = [mean(active_cuts_df[i][1:end, :num_active_cuts]) for i in 1:length(active_cuts_df)]
                  else
                        active_cuts = [0.0 for _ in time_list_oos]
                  end
                  filename = joinpath(folder, "approx_values.csv")
                  if isfile(filename)
                        LB_df = CSV.read(filename, DataFrame)
                        LB = [maximum(LB_df[LB_df[:, :time] .<= t, :approx_value])/1000 for t in time_list_oos]
                  else
                        LB = [0.0 for _ in time_list_oos]
                  end
                  results[discount_factor][(seed, label)] = (oos_mean, oos_std, active_cuts, LB)
            end
      end
end


results_mean = Dict()
for discount_factor in discount_factor_list
    results_mean[discount_factor] = Dict()
    for label in keys(folder_list)
      all_oos_mean = [results[discount_factor][(seed, label)][1] for seed in seed_list]
      all_oos_std = [results[discount_factor][(seed, label)][2] for seed in seed_list]
      all_active_cuts = [results[discount_factor][(seed, label)][3] for seed in seed_list]
      all_LB = [results[discount_factor][(seed, label)][4] for seed in seed_list]

      oos_mean_matrix = hcat(all_oos_mean...)  # 10 x 10 matrix (10 iterations x 10 seeds)
      oos_std_matrix = hcat(all_oos_std...)  # 10 x 10 matrix (10 iterations x 10 seeds)

      oos_mean = mean(oos_mean_matrix, dims=2)[:]
      oos_std = mean(oos_std_matrix, dims=2)[:]

      active_cuts_matrix = hcat(all_active_cuts...)  # 10 x 10 matrix (10 iterations x 10 seeds)
      active_cuts_mean = mean(active_cuts_matrix, dims=2)[:]

      LB_matrix = hcat(all_LB...)  # 10 x 10 matrix (10 iterations x 10 seeds)
      LB_mean = mean(LB_matrix, dims=2)[:]

      results_mean[discount_factor][label] = (oos_mean, oos_std, active_cuts_mean, LB_mean)
      end
end

fontsize = 20
p1=plot(label = false, xlabel="Time (s)", ylabel ="Cost",guidefontsize=fontsize, tickfontsize=fontsize, legendfontsize=fontsize, margin=10Plots.mm)
p2=plot(label = false, xlabel="Time (s)", ylabel ="Cost",guidefontsize=fontsize, tickfontsize=fontsize, legendfontsize=fontsize, margin=10Plots.mm)
p3=plot(label = false, xlabel="Time (s)", ylabel ="Cost",guidefontsize=fontsize, tickfontsize=fontsize, legendfontsize=fontsize, margin=10Plots.mm)
p=[p1, p2, p3]

color = 1
for (label, _) in folder_list
      for (i, discount_factor) in enumerate(discount_factor_list)
            N = N_list[discount_factor]
            tcrit = quantile(TDist(N-1), 0.975)
            oos_mean = results_mean[discount_factor][label][1]
            oos_mean_up = results_mean[discount_factor][label][1] + tcrit * results_mean[discount_factor][label][2] / sqrt(N)
            oos_mean_down = results_mean[discount_factor][label][1] - tcrit * results_mean[discount_factor][label][2] / sqrt(N)
            plot!(p[i], time_list_oos, oos_mean,label=false, ribbon=(oos_mean - oos_mean_down, oos_mean_up - oos_mean), linewidth=3, fillalpha=0.3, color = color)
            active_cuts = results_mean[discount_factor][label][3]
            # plot!(p[i], time_list_oos, active_cuts,label=false, linewidth=3, color = color)
      end
      color += 1
end

pleg = plot(
    [NaN], [NaN],
    label = "Cyclic SDDP",
    linewidth = 3,
    color = 1,
    framestyle = :none,
    legend = :bottom,
    legend_columns = 2,
    legendfontsize = fontsize,
    ticks = false,
    grid = false,
)
plot!(
    pleg,
    [NaN], [NaN],
    label = "RV-SDDP",
    linewidth = 3,
    color = 2,
)

plot(
    p1,
    p2,
    p3,
    pleg,
    layout = @layout([a b c; d{0.2h}]),
    size = (2100, 900),
)

In [ ]:
folder_list = Dict("RV-SDDP" => "shift_update_random_forward_0_parallel_1", "Cyclic SDDP" => "no_shift_0_parallel_1", "Periodic RV-SDDP" => "shift_update_random_forward_1_parallel_1", "Periodic SDDP" => "no_shift_1_parallel_1", "RV-SDDP_10"=> "shift_update_random_forward_0_parallel_10", "Cyclic SDDP_10"=> "no_shift_0_parallel_10")
results = Dict()
seed_list = 1:10
time_max = 36000
parallel = 1
time_list_oos = [600,  3600, 36000]
discount_factor=0.995
period = 12
N=5000
TimeHorizon = 12*Int(ceil(log(0.001)/(12*log(discount_factor))))
time_list_value = 300:300:36000
for seed in seed_list
      for (label,folder) in folder_list
            TimeHorizon = 12*Int(ceil(log(0.001)/(12*log(discount_factor))))
            folder = "results_msppy/$(folder)/$(discount_factor)/seed_$(seed)_time_$(time_max)"
            if isdir("$(folder)/oos")
                  oos_df = [CSV.read(joinpath("$(folder)/oos", "oos_horizon_$(time)_$(TimeHorizon)_$N.csv"), DataFrame) for time in time_list_oos]
                  oos_mean = [mean(oos_df[i][1:N, :oos_horizon])/1000 for i in 1:length(oos_df)]
                  oos_std = [std(oos_df[i][1:N, :oos_horizon])/1000 for i in 1:length(oos_df)]
            else
                  oos_mean = [0.0 for _ in time_list_oos]
                  oos_std = [0.0 for _ in time_list_oos]
            end
            if isdir("$(folder)/active_cuts")
                  active_cuts_df = [CSV.read(joinpath("$(folder)/active_cuts", "time_$(time).csv"), DataFrame) for time in time_list_oos]
                  active_cuts = [mean(active_cuts_df[i][1:end, :num_active_cuts]) for i in 1:length(active_cuts_df)]
                  df_cuts = CSV.read("$(folder)/cuts.csv", DataFrame)
                  nb_cuts = [sum([1 for row in eachrow(df_cuts) if row.time <= t])/period for t in time_list_oos]
            else
                  active_cuts = [0.0 for _ in time_list_oos]
                  nb_cuts = [0.0 for _ in time_list_oos]
            end
            filename = joinpath(folder, "approx_values.csv")
            if isfile(filename)
                  LB_df = CSV.read(filename, DataFrame)
                  LB = [maximum(LB_df[LB_df[:, :time] .<= t, :approx_value])/1000 for t in time_list_oos]
            else
                  LB = [0.0 for _ in time_list_oos]
            end
            results[(seed, label)] = (oos_mean, oos_std, active_cuts, nb_cuts, LB)
      end
end


results_mean = Dict()
for label in keys(folder_list)
      all_oos_mean = [results[(seed, label)][1] for seed in seed_list]
      all_oos_std = [results[(seed, label)][2] for seed in seed_list]
      all_active_cuts = [results[(seed, label)][3] for seed in seed_list]
      all_nb_cuts = [results[(seed, label)][4] for seed in seed_list]
      all_LB = [results[(seed, label)][5] for seed in seed_list]

      oos_mean_matrix = hcat(all_oos_mean...)  # 10 x 10 matrix (10 iterations x 10 seeds)
      oos_std_matrix = hcat(all_oos_std...)  # 10 x 10 matrix (10 iterations x 10 seeds)

      oos_mean = mean(oos_mean_matrix, dims=2)[:]
      oos_std = mean(oos_std_matrix, dims=2)[:]

      active_cuts_matrix = hcat(all_active_cuts...)  # 10 x 10 matrix (10 iterations x 10 seeds)
      active_cuts_mean = mean(active_cuts_matrix, dims=2)[:]

      nb_cuts_matrix = hcat(all_nb_cuts...)  # 10 x 10 matrix (10 iterations x 10 seeds)
      nb_cuts_mean = mean(nb_cuts_matrix, dims=2)[:]

      LB_matrix = hcat(all_LB...)  # 10 x 10 matrix (10 iterations x 10 seeds)
      LB_mean = mean(LB_matrix, dims=2)[:]

      results_mean[label] = (oos_mean, oos_std, active_cuts_mean, nb_cuts_mean, LB_mean)
end
min_oos_mean = minimum([cost for label in keys(folder_list) for cost in results_mean[label][1]])
for label in keys(folder_list)
    println(1000*results_mean[label][1]/min_oos_mean)
    results_mean[label] = (1000*results_mean[label][1]./min_oos_mean, 1000*results_mean[label][2]./min_oos_mean, results_mean[label][3], results_mean[label][4], 1000*results_mean[label][5]/min_oos_mean)
end

In [ ]:
label_list = ["RV-SDDP", "Cyclic SDDP", "Periodic RV-SDDP", "Periodic SDDP", "RV-SDDP_10", "Cyclic SDDP_10"]
function latex_escape(s::AbstractString)
    return replace(s, "_" => "\\_")
end

function fmt_int(x)
    return string(round(Int, x))
end

function write_overleaf_table(
    results_mean,
    time_list_oos;
    discount_factor = 0.99,
    filename = "results_table_beta_$(discount_factor).txt",
    caption = "Comparison of methods.",
    label = "tab:comparison_beta",
    N = 1
)
    open(filename, "w") do io
        println(io, "\\begin{table}[ht]")
        println(io, "\\centering")
        println(io, "\\footnotesize")
        println(io, "\\setlength{\\tabcolsep}{4pt}")
        println(io, "\\caption{$caption Costs are reported as mean \$\\pm\$ standard deviation. A/T denotes the number of active cuts over total cuts (on average per stage).}")
        println(io, "\\label{$label}")
        println(io, "\\begin{tabular}{lcc|cc|cc}")
        println(io, "\\toprule")

        println(io,
            "& \\multicolumn{2}{c|}{$(time_list_oos[1]) s} " *
            "& \\multicolumn{2}{c|}{$(time_list_oos[2]) s} " *
            "& \\multicolumn{2}{c}{$(time_list_oos[3]) s} \\\\"
        )

        println(io, "\\cmidrule(r){2-3}")
        println(io, "\\cmidrule(r){4-5}")
        println(io, "\\cmidrule(l){6-7}")

        println(io,
            "Method " *
            "& Cost & Cuts (A/T) " *
            "& Cost & Cuts (A/T) " *
            "& Cost & Cuts (A/T) \\\\"
        )

        println(io, "\\midrule")

        tcrit = quantile(TDist(N-1), 0.975)
        for label_method in label_list
            oos_mean = results_mean[label_method][1]
            oos_std = results_mean[label_method][2]
            active_cuts_mean = results_mean[label_method][3]
            total_cuts_mean = results_mean[label_method][4]
            LB_mean = results_mean[label_method][5]

            method_name = latex_escape(string(label_method))

            row = method_name

            for i in 1:3
                cost = fmt_int(oos_mean[i])
                # cost = string(round(oos_mean[i]; digits=1))
                std = fmt_int(Int(round(tcrit * oos_std[i]/ sqrt(N))))
                # std = string(round(tcrit * oos_std[i]/ sqrt(N); digits=1))
                active = fmt_int(active_cuts_mean[i])
                total = fmt_int(total_cuts_mean[i])
                LB = fmt_int(LB_mean[i])
                # LB = string(round(LB_mean[i]; digits=1))
                if label_method in ["Periodic RV-SDDP", "Periodic SDDP"]
                    if round(Int,oos_mean[i]) <= minimum([round(Int,results_mean[l][1][i]) for l in ["Periodic RV-SDDP", "Periodic SDDP"]])
                        cost_str = "\$ \\bm{$(cost) \\pm $(std)}\$"
                    else 
                        cost_str = "\$$(cost) \\pm $(std)\$"
                    end
                    if round(Int, active_cuts_mean[i]) >= maximum([round(Int,results_mean[l][3][i]) for l in ["Periodic RV-SDDP", "Periodic SDDP"]])
                        active_str = "\\bm{$(active)}"
                    else 
                        active_str = active
                    end
                elseif label_method in ["Cyclic SDDP", "RV-SDDP"]
                    if round(Int,oos_mean[i]) <= minimum([round(Int,results_mean[l][1][i]) for l in ["Cyclic SDDP", "RV-SDDP"]])
                        cost_str = "\$ \\bm{$(cost) \\pm $(std)}\$"
                    else
                        cost_str = "\$$(cost) \\pm $(std)\$"
                    end
                    if round(Int, active_cuts_mean[i]) >= maximum([round(Int,results_mean[l][3][i]) for l in ["Cyclic SDDP", "RV-SDDP"]])
                        active_str = "\\bm{$(active)}"
                    else 
                        active_str = active
                    end
                elseif label_method in ["RV-SDDP_10", "Cyclic SDDP_10"]
                    if round(Int,oos_mean[i]) <= minimum([round(Int,results_mean[l][1][i]) for l in ["RV-SDDP_10", "Cyclic SDDP_10"]])
                        cost_str = "\$ \\bm{$(cost) \\pm $(std)}\$"
                    else
                        cost_str = "\$$(cost) \\pm $(std)\$"
                    end
                    if round(Int, active_cuts_mean[i]) >= maximum([round(Int,results_mean[l][3][i]) for l in ["RV-SDDP_10", "Cyclic SDDP_10"]])
                        active_str = "\\bm{$(active)}"
                    else 
                        active_str = active
                    end
                end


                if label_method in ["Cyclic SDDP", "Periodic SDDP", "Cyclic SDDP_10"]
                    row *= " & \\makecell[c]{$(cost_str) \\\\{\\scriptsize ($(LB))}} & \$$(active_str) / $(total)\$"
                else
                    row *= " & $(cost_str) & \$$(active_str) / $(total)\$"
                end
            end

            row *= " \\\\"
            if label_method in ["Cyclic SDDP", "Periodic SDDP"]
                row *= "[1ex] \\cdashlinelr{1-7} \\noalign{\\vskip 0.3ex}"
            end
            println(io, row)
        end

        println(io, "\\bottomrule")
        println(io, "\\end{tabular}")
        println(io, "\\end{table}")
    end
end

In [ ]:
write_overleaf_table(
    results_mean,
    time_list_oos;
    discount_factor = discount_factor,
    filename = "results_msppy/table_beta_$(discount_factor).txt",
    caption = "Comparison of methods for \$\\beta=$(discount_factor)\$.",
    label = "tab:comparison_beta_$(replace(string(discount_factor), "." => ""))",
    N = N
)